In [1]:
import sys
sys.path.insert(0, '../')
import logging
logging.basicConfig(level=logging.ERROR)

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

from cryptotrader.envs.trading import PaperTradingEnvironment, PaperTradingDataFeed
from cryptotrader.agents import apriori
from cryptotrader.exchange_api.poloniex import Poloniex
from cryptotrader.envs.utils import make_balance

from time import time, sleep

from bokeh.io import output_notebook
output_notebook()
%matplotlib inline

Loading BokehJS ...

In [2]:
obs_steps = 30
period = 5
pairs = ["USDT_BTC", "USDT_ETH", "USDT_LTC", "USDT_XRP", "USDT_XMR", "USDT_ETC", "USDT_ZEC", "USDT_DASH"]
init_funds = make_balance(crypto=0.0, fiat=100.0, pairs=pairs)

tapi = Poloniex()
tapi = PaperTradingDataFeed(tapi, period, pairs, init_funds)

env = PaperTradingEnvironment(period, obs_steps, tapi, 'dev_tapi_env')
env.add_pairs(pairs)
env.fiat = 'USDT'

# env.set_email('email@gmail.com', 'password')

obs = env.reset()

INFO:dev_tapi_env:[Trading Environment initialization]
Trading Environment Initialized!



In [3]:
agent = apriori.TestAgent(env.get_observation(True).shape)
agent.trade(env, verbose=True)

Executing paper trading with 5 min frequency.
Initial portfolio value: 100 fiat units.
>> step 5, Uptime: 0 days 00:16:34.395499, Crypto prices: ['USDT_BTC: 7024.00000000', 'USDT_ETH: 286.44078048', 'USDT_LTC: 54.11550588', 'USDT_XRP: 0.21170000', 'USDT_XMR: 83.91600000', 'USDT_ETC: 10.19999874', 'USDT_ZEC: 219.64857679', 'USDT_DASH: 266.33428292'], Last action: {'BTC': 0.12519838, 'ETH': 0.12376973, 'LTC': 0.12421942, 'XRP': 0.12483188, 'XMR': 0.12469386, 'ETC': 0.12698744, 'ZEC': 0.12447554, 'DASH': 0.12582374, 'USDT': 2e-08, 'online': 1.0}, Portval: 100.20                                                                                               
Keyboard Interrupt: Stoping cryptotrader
Elapsed steps: 5
Uptime: 0 days 00:16:39.272050
Final Portval: 100.2043727738192218



In [4]:
env.plot_results();


################### > Portifolio Performance Analysis < ###################

Portifolio excess Sharpe:                 0.036154
Portifolio / Benchmark Sharpe ratio:      11.920292 / 6.276632
Portifolio / Benchmark Omega ratio:       nan / 3.432622
Portifolio / Benchmark max drawdown:      0.000000 / -0.209691


In [5]:
env.portfolio_df

,BTC,ETH,LTC,XRP,XMR,ETC,ZEC,DASH,USDT,portval
2017-11-03 00:24:28.392220+00:00,0E-8,0E-8,0E-8,0E-8,0E-8,0E-8,0E-8,0E-8,100.00000000,NaN
2017-11-03 00:24:43.679516+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00000000
2017-11-03 00:24:43.704744+00:00,0.00178848,0.04336220,0.23035601,58.53875438,0.14896954,1.24937373,0.05687054,0.04740970,0E-8,99.74999710
2017-11-03 00:25:01.596832+00:00,0.00178848,0.04336220,0.23035600,58.53875268,0.14896954,1.24937370,0.05687054,0.04740970,0.00000199,99.74999789
2017-11-03 00:30:07.504240+00:00,0.00178848,0.04336220,0.23035600,58.53875229,0.14896955,1.24937374,0.05687054,0.04740970,9.9E-7,100.13548052
2017-11-03 00:35:21.255975+00:00,0.00178848,0.04336220,0.23035599,58.53875202,0.14896955,1.24937376,0.05687054,0.04740970,0.00000100,100.20103387
2017-11-03 00:40:17.946395+00:00,0.00178848,0.04336220,0.23035598,58.53875158,0.14896955,1.24937372,0.05687054,0.04740970,0.00000199,100.35331385


In [6]:
pvec = env.get_sampled_portfolio()
pvec

,BTC,ETH,LTC,XRP,XMR,ETC,ZEC,DASH,USDT,portval
2017-11-03 00:20:00+00:00,0.00178848,0.04336220,0.23035601,58.53875438,0.14896954,1.24937373,0.05687054,0.04740970,0E-8,99.74999710
2017-11-03 00:25:00+00:00,0.00178848,0.04336220,0.23035600,58.53875268,0.14896954,1.24937370,0.05687054,0.04740970,0.00000199,99.74999789
2017-11-03 00:30:00+00:00,0.00178848,0.04336220,0.23035600,58.53875229,0.14896955,1.24937374,0.05687054,0.04740970,9.9E-7,100.13548052
2017-11-03 00:35:00+00:00,0.00178848,0.04336220,0.23035599,58.53875202,0.14896955,1.24937376,0.05687054,0.04740970,0.00000100,100.20103387
2017-11-03 00:40:00+00:00,0.00178848,0.04336220,0.23035598,58.53875158,0.14896955,1.24937372,0.05687054,0.04740970,0.00000199,100.35331385


In [7]:
env.get_history(pvec.index[0], pvec.index[-1])

USDT_BTC                                \
                                    open           high            low   
2017-11-03 00:20:00+00:00  6959.11850717  6980.06350285  6942.98165494   
2017-11-03 00:25:00+00:00  6979.98899996  6998.03015545  6958.00000001   
2017-11-03 00:30:00+00:00  6990.30070520  7010.00000000  6989.05895379   
2017-11-03 00:35:00+00:00  7005.98902720  7043.35726519  7000.00000050   
2017-11-03 00:40:00+00:00  7024.73253112  7024.73253112  7011.95443426   

                                                               USDT_ETH  \
                                   close           volume          open   
2017-11-03 00:20:00+00:00  6979.99999993  334234.47599701  285.99999999   
2017-11-03 00:25:00+00:00  6996.99999999  216315.40099226  287.54885107   
2017-11-03 00:30:00+00:00  7000.00000050  397166.65341576  288.00000153   
2017-11-03 00:35:00+00:00  7024.00000000  157995.52772049  287.17098538   
2017-11-03 00:40:00+00:00  7011.95443426    3765.09581069  286.44077999   

                                                                     \
                                   high           low         close   
2017-11-03 00:20:00+00:00  288.00000000  284.30000042  288.00000000   
2017-11-03 00:25:00+00:00  289.00000000  286.34114622  289.00000000   
2017-11-03 00:30:00+00:00  289.00000000  287.26425459  287.58600000   
2017-11-03 00:35:00+00:00  288.85299996  286.44078038  286.44078048   
2017-11-03 00:40:00+00:00  286.44077999  285.53200000  285.53200000   

                                                ...            USDT_ZEC  \
                                    volume      ...                open   
2017-11-03 00:20:00+00:00  245382.55642246      ...        218.55241688   
2017-11-03 00:25:00+00:00   83471.82219592      ...        219.00000000   
2017-11-03 00:30:00+00:00   23191.74963297      ...        218.07852367   
2017-11-03 00:35:00+00:00   29113.00103166      ...        219.24795499   
2017-11-03 00:40:00+00:00    2204.28345209      ...        219.64857679   

                                                                     \
                                   high           low         close   
2017-11-03 00:20:00+00:00  219.24795350  218.55241688  219.00000000   
2017-11-03 00:25:00+00:00  219.00000005  218.07117042  218.07852362   
2017-11-03 00:30:00+00:00  218.07852367  218.07852362  218.07852362   
2017-11-03 00:35:00+00:00  219.64857679  219.24795499  219.64857679   
2017-11-03 00:40:00+00:00  219.90000000  219.64857679  219.64857679   

                                             USDT_DASH                \
                                  volume          open          high   
2017-11-03 00:20:00+00:00  7179.03676659  263.00000006  263.44999906   
2017-11-03 00:25:00+00:00  5781.25284826  263.45000005  265.44999895   
2017-11-03 00:30:00+00:00    25.41932648  265.44999894  265.78000000   
2017-11-03 00:35:00+00:00   161.77519774  265.78000000  266.33428292   
2017-11-03 00:40:00+00:00   618.04057255  265.78000000  266.33428292   

                                                                      
                                    low         close         volume  
2017-11-03 00:20:00+00:00  263.00000006  263.44999904  1426.72968647  
2017-11-03 00:25:00+00:00  263.45000005  265.44999895    63.01490375  
2017-11-03 00:30:00+00:00  265.44999894  265.44999895  2492.39902782  
2017-11-03 00:35:00+00:00  265.44999975  266.33428292   561.24677947  
2017-11-03 00:40:00+00:00  265.44999975  266.33428292   561.24677947  

[5 rows x 40 columns]